# 🎓 Mini Project: Income Prediction using Census Demographics Data

**Course:** Data Science  
**Project Type:** Machine Learning Model Training  
**Domain:** Social / Demographics  

---

## 📌 Problem Statement

Can we predict whether a person earns **more than $50K/year** based on their demographic information such as age, education, occupation, and marital status?

This is a **binary classification** problem. We will use the famous **Adult Census Income Dataset** (also known as the "Census Income" dataset) from the UCI Machine Learning Repository.

---

## 📂 Dataset

- **Source:** UCI Machine Learning Repository (also available on Kaggle)
- **Link:** https://archive.ics.uci.edu/ml/datasets/adult
- **Records:** ~48,000 individuals
- **Features:** Age, Education, Occupation, Marital Status, Race, Gender, Hours-per-week, etc.
- **Target:** Income ( `<=50K` or `>50K` )

---

## 🗂️ Project Workflow

1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Model Training (Logistic Regression & Decision Tree)
6. Model Evaluation
7. Conclusion


---
## 1. 📦 Import Libraries

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries imported successfully!")

---
## 2. 📥 Load Dataset

In [ ]:
# Load the Adult Census Income dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'gender',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv(url, names=column_names, sep=',\s*', engine='python', na_values='?')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

---
## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# Check missing values
print("🔎 Missing Values:")
print(df.isnull().sum())

In [ ]:
# Target variable distribution
plt.figure(figsize=(7, 4))
ax = sns.countplot(x='income', data=df, palette='Set2')
plt.title('Income Distribution (Target Variable)', fontsize=14, fontweight='bold')
plt.xlabel('Income Category')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

print("\nValue Counts:")
print(df['income'].value_counts())

In [ ]:
# Age distribution by income
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='income', kde=True, bins=30, palette='Set1')
plt.title('Age Distribution by Income', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Education vs Income
plt.figure(figsize=(12, 5))
edu_income = df.groupby(['education', 'income']).size().unstack()
edu_income.plot(kind='bar', stacked=True, colormap='Set2', figsize=(14, 5))
plt.title('Education Level vs Income', fontsize=14, fontweight='bold')
plt.xlabel('Education')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Income')
plt.tight_layout()
plt.show()

In [ ]:
# Gender vs Income
plt.figure(figsize=(7, 4))
sns.countplot(x='gender', hue='income', data=df, palette='pastel')
plt.title('Gender vs Income', fontsize=14, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Hours per week distribution
plt.figure(figsize=(10, 4))
sns.boxplot(x='income', y='hours_per_week', data=df, palette='Set3')
plt.title('Hours Per Week vs Income', fontsize=14, fontweight='bold')
plt.xlabel('Income')
plt.ylabel('Hours Per Week')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features)
plt.figure(figsize=(8, 6))
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Numerical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. 🛠️ Data Preprocessing

In [ ]:
# Drop rows with missing values
df_clean = df.dropna().copy()
print(f"✅ Rows after dropping nulls: {df_clean.shape[0]}")

In [ ]:
# Drop 'fnlwgt' (final weight - not useful for prediction) and 'education' (redundant with education_num)
df_clean = df_clean.drop(columns=['fnlwgt', 'education'])

# Encode the target variable: <=50K → 0, >50K → 1
df_clean['income'] = df_clean['income'].map({'<=50K': 0, '>50K': 1})

print("✅ Target encoded:")
print(df_clean['income'].value_counts())

In [ ]:
# Label encode all remaining categorical columns
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

print("\n✅ Encoding complete. Sample:")
df_clean.head(3)

In [ ]:
# Split features and target
X = df_clean.drop('income', axis=1)
y = df_clean['income']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Training samples: {X_train.shape[0]}")
print(f"✅ Testing samples:  {X_test.shape[0]}")

In [ ]:
# Feature scaling (needed for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Features scaled successfully!")

---
## 5. 🤖 Model Training

### Model 1 — Logistic Regression
A simple, interpretable linear model often used as a baseline for binary classification.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_preds = lr_model.predict(X_test_scaled)

print("✅ Logistic Regression trained!")
print(f"\n📈 Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, lr_preds, target_names=['<=50K', '>50K']))

In [ ]:
# Confusion Matrix - Logistic Regression
cm_lr = confusion_matrix(y_test, lr_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Model 2 — Decision Tree Classifier
A non-linear, tree-based model that is easy to visualize and understand.

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)  # Decision Trees don't require scaling

# Predictions
dt_preds = dt_model.predict(X_test)

print("✅ Decision Tree trained!")
print(f"\n📈 Accuracy: {accuracy_score(y_test, dt_preds):.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, dt_preds, target_names=['<=50K', '>50K']))

In [ ]:
# Confusion Matrix - Decision Tree
cm_dt = confusion_matrix(y_test, dt_preds)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['<=50K', '>50K'])
disp2.plot(cmap='Greens')
plt.title('Confusion Matrix — Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. 📊 Model Evaluation & Comparison

In [ ]:
# Side-by-side accuracy comparison
models = ['Logistic Regression', 'Decision Tree']
accuracies = [
    accuracy_score(y_test, lr_preds),
    accuracy_score(y_test, dt_preds)
]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accuracies, color=['steelblue', 'seagreen'], width=0.4, edgecolor='black')
plt.ylim(0.7, 0.9)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Decision Tree)
feature_importance = pd.Series(dt_model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=feature_importance.values, y=feature_importance.index, palette='viridis')
plt.title('Feature Importances — Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feature_importance.head())

In [ ]:
# Summary Table
from sklearn.metrics import precision_score, recall_score, f1_score

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy':  [accuracy_score(y_test, lr_preds), accuracy_score(y_test, dt_preds)],
    'Precision': [precision_score(y_test, lr_preds), precision_score(y_test, dt_preds)],
    'Recall':    [recall_score(y_test, lr_preds), recall_score(y_test, dt_preds)],
    'F1-Score':  [f1_score(y_test, lr_preds), f1_score(y_test, dt_preds)]
})

summary = summary.set_index('Model').round(4)
print("\n📊 Model Performance Summary:")
summary

---
## 7. ✅ Conclusion

### What We Did
- Loaded the **Adult Census Income Dataset** with ~48,000 real-world demographic records.
- Performed **Exploratory Data Analysis (EDA)** to understand how features like age, education, gender, and occupation relate to income.
- Preprocessed data: handled missing values, encoded categorical variables, and scaled features.
- Trained two ML models: **Logistic Regression** and **Decision Tree Classifier**.
- Evaluated and compared both models using Accuracy, Precision, Recall, and F1-Score.

### Key Findings
- **Education level, age, and capital gain** are among the strongest predictors of income.
- **Males** in this dataset are more likely to earn >50K than females, reflecting historical demographic patterns.
- People working **more hours per week** tend to have higher incomes.
- Both models achieved **~82–85% accuracy**, which is strong for a baseline.

### Future Improvements
- Try more powerful models: **Random Forest**, **Gradient Boosting**, or **XGBoost**.
- Handle class imbalance using **SMOTE** or class weights.
- Perform **hyperparameter tuning** using GridSearchCV.
- Apply **cross-validation** for more robust evaluation.

---
*Project completed as part of Data Science coursework.*